# 01 — Data audit and evaluation protocol

This notebook loads the supplied CSV files and retained audit evidence. It performs no model fitting or hyperparameter selection. The later notebooks use the same sources and saved predictions. All paths are relative to the project root.

Part 1 predicts five possibly overlapping stopping reasons from provided 150-dimensional embeddings. Only observed labels supervise the models. Part 2 predicts success separately for phase I/II/III trial rows using criteria, disease, and drug text; stopping reasons are reserved for retrospective error analysis.


In [2]:
from pathlib import Path
import sys, os, json, tempfile
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p/'models/selection.json').is_file() and (p/'tools/src').is_dir())
for module_dir in (ROOT/'tools/src', ROOT/'report/scripts'):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))
os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir())/'midterm-report-mpl'))
import numpy as np
import pandas as pd
from IPython.display import display, Image, Code
from midterm_evaluation import REASONS, dataset_path
RUN_TRAINING = False  # Explicit opt-in only. Run All uses saved evidence and does not train.
print('Consolidated project | mode: saved evidence | training:', RUN_TRAINING)


In [3]:
datasets = {(task,split): pd.read_csv(dataset_path(ROOT,task,split))
            for task in ('part1','part2') for split in ('train','val','test')}
overview=[]
for (task,split),frame in datasets.items():
    observed = int(frame[list(REASONS)].isin([0,1]).all(axis=1).sum()) if task=='part1' and split!='test' else None
    overview.append({'task':task,'split':split,'rows':len(frame),'unique_IDs':frame.ID.nunique(),
                     'unique_trials':frame.trial_id.nunique() if 'trial_id' in frame else frame.ID.nunique(),
                     'observed_reason_rows':observed,'year_min':frame.start_year.min(),'year_max':frame.start_year.max()})
    assert frame.ID.is_unique
display(pd.DataFrame(overview))
for task in ('part1','part2'):
    for left,right in [('train','val'),('train','test'),('val','test')]:
        assert not set(datasets[task,left].ID)&set(datasets[task,right].ID)
print('ID split intersections: zero')


,task,split,rows,unique_IDs,unique_trials,observed_reason_rows,year_min,year_max
0,part1,train,1171,1171,1171,157.0,1993.0,2010.0
1,part1,val,78,78,78,78.0,1999.0,2009.0
2,part1,test,312,312,312,NaN,2010.0,2016.0
3,part2,train,8037,8037,7535,NaN,1993.0,2013.0
4,part2,val,1013,1013,901,NaN,1995.0,2013.0
5,part2,test,1500,1500,1442,NaN,2014.0,2021.0


In [4]:
part1_train=datasets['part1','train']
gold=part1_train.loc[part1_train[list(REASONS)].isin([0,1]).all(axis=1)]
supports=pd.DataFrame({'reason':REASONS,'training_positive':gold[list(REASONS)].sum().to_numpy(),
                      'validation_positive':datasets['part1','val'][list(REASONS)].sum().to_numpy()})
display(supports)
display(pd.read_csv(ROOT/'data/audit/tables/part1_label_cardinality.csv'))
display(pd.read_csv(ROOT/'data/audit/tables/year_summary.csv'))


,reason,training_positive,validation_positive
0,Safety,16,9
1,Funding,8,4
2,Efficacy,17,14
3,Recruitment,101,50
4,Other,25,10


,group,positive_reasons_per_trial,trials,trial_rate
0,train_visible,0,0,0.000000
1,train_visible,1,147,0.936306
2,train_visible,2,10,0.063694
3,train_visible,3,0,0.000000
4,train_visible,4,0,0.000000
5,train_visible,5,0,0.000000
6,validation,0,0,0.000000
7,validation,1,69,0.884615
8,validation,2,9,0.115385
9,validation,3,0,0.000000


,group,rows,known_year_rows,missing_year_rows,min,q25,median,q75,max
0,part1_train_visible,157,157,0,1993.0,1999.0,2001.0,2003.0,2010.0
1,part1_train_masked,1014,1014,0,2003.0,2005.0,2007.0,2008.0,2010.0
2,part1_val,78,78,0,1999.0,2005.0,2006.0,2008.0,2009.0
3,part1_test,312,312,0,2010.0,2010.0,2011.0,2011.0,2016.0
4,outcome_train,8037,8016,21,1993.0,2005.0,2007.0,2010.0,2013.0
5,outcome_val,1013,1007,6,1995.0,2005.0,2007.0,2010.0,2013.0
6,outcome_test,1500,1500,0,2014.0,2014.0,2015.0,2016.0,2021.0


## Validation scope and leakage controls

- Part 1 historical validation uses 157 labeled training trials and evaluates 78 validation trials. The final ridge submission uses all 235 visible labels; its validation predictions are **not** used as a holdout score.
- Repeated Part 1 OOF uses four grouped folds and seeds 42, 137, and 2026. Near-identical embeddings are connected into groups at raw Euclidean distance ≤ 1e-4. A held group is removed from both labeled and unlabeled fitting pools. Inner folds select models; outer folds estimate the selection procedure.
- Part 2 cross-validation groups all phase rows sharing `trial_id`, keeping their vocabularies and fits inside the training fold. Primary log loss is the mean of the three phase losses. AP uses sklearn average precision, not trapezoidal PR integration.
- These labels and earlier public results have been repeatedly inspected. Resampling and historical-year tests are useful diagnostics, not untouched external evidence. Public leaderboard scores are recorded separately.


In [5]:
groups=pd.read_csv(ROOT/'models/part1/03_kernel_ridge/evidence/selection_procedure/groups.csv')
display(groups.head(8))
display(pd.read_csv(ROOT/'data/audit/tables/within_part_split_overlap.csv'))
protocol=json.loads((ROOT/'data/audit/protocol/protocol.json').read_text())
display(protocol)


,ID,source,start_year,group
0,TR04011,train,2002,0
1,TR09312,train,1999,0
2,TR01012,train,1999,1
3,TR08630,train,2002,2
4,TR09705,train,2006,3
5,TR05389,train,2001,4
6,TR08552,train,2001,5
7,TR08249,train,2001,6


,part,split_a,split_b,shared_unique_trials
0,Part1,train,val,0
1,Part1,train,test,0
2,Part1,val,test,0
3,Part2,train,val,0
4,Part2,train,test,0
5,Part2,val,test,0


{'version': '1.0',
 'seed': 42,
 'selection_split': 'provided_validation',
 'part1': {'primary_metric': 'f1_samples',
  'higher_is_better': True,
  'report': ['f1_samples',
   'f1_macro',
   'f1_per_reason',
   'positive_support_per_reason']},
 'part2': {'primary_metric': 'log_loss_equal_phase',
  'higher_is_better': False,
  'report_per_phase': ['f1',
   'roc_auc',
   'average_precision',
   'log_loss',
   'positive_support',
   'negative_support'],
  'positive_class': 1,
  'PR_AUC_definition': 'sklearn average_precision_score; no trapezoidal PR integration'},
 'decision_rule': 'probability >= threshold',
 'initial_threshold': 0.5,
 'f1_zero_division': 0,
 'log_loss_clipping_epsilon': 1e-15,
 'undefined_auc_policy': 'ROC-AUC: missing for single-class/no-row phases; AP: missing for no-positive/no-row phases; JSON uses null.',
 'preprocessing': 'Fit each model, scaler, vocabulary, clusterer, and imputer using training data only; transform validation/test.',
 'part2_forbidden_inputs': ['

In [6]:
selection=json.loads((ROOT/'models/selection.json').read_text())
display(pd.DataFrame([{'part':part,'public_score':selection[part]['best_public']['public_score'],
                       'submission':selection[part]['best_public']['submission']} for part in ('part1','part2')]))
print('Private leaderboard scores are unknown. Public scores are user-reported screenshot evidence.')


,part,public_score,submission
0,part1,0.88568,models/part1/03_kernel_ridge/submission.csv
1,part2,0.59011,models/part2/03_phase_shared/submission.csv
